## Prepare a full knowledge graph dataset


We need to prepare the training, validation and external test datasets. We will use the training dataset to train the model and the test dataset to evaluate the model for all KGE models.

### [Require to Modify According to Your Situation] Prepare all relation files

We design a strategy to allow users to integrate their expected relation files for different purposes. Such as you might want to include the `malacards_mecfs` dataset when you want to train a model for ME/CFS.

In [ ]:
ls -lh /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmp29yfyc32/

In [1]:
import os
import pandas as pd
import subprocess

temp_dir = "/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmp29yfyc32"
# 更稳妥的话，改成你刚复制出来的 checkpoints 路径

entity_file = "/Users/zhuzhixing/KG/biomedgps-data/graph_data/entities.tsv"
kg_file = os.path.join(temp_dir, "knowledge_graph.tsv")
kg_clean_file = os.path.join(temp_dir, "knowledge_graph_clean.tsv")
annotated_kg_file = os.path.join(temp_dir, "annotated_knowledge_graph.tsv")

outputdir = "/Volumes/T7/biomedgps_output"
os.makedirs(outputdir, exist_ok=True)

In [17]:
print("Check files:")
for f in [kg_file, kg_clean_file, annotated_kg_file, entity_file]:
    print(f, "->", os.path.exists(f))

Check files:
/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmp29yfyc32/knowledge_graph.tsv -> True
/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmp29yfyc32/knowledge_graph_clean.tsv -> True
/var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmp29yfyc32/annotated_knowledge_graph.tsv -> True
/Users/zhuzhixing/KG/biomedgps-data/graph_data/entities.tsv -> True


### Extract valid entities from the knowledge graph

In [4]:
entities = pd.read_csv(entity_file, sep="\t", low_memory=False)
knowledge_graph = pd.read_csv(kg_file, sep="\t", low_memory=False)

source_entities = knowledge_graph[["source_id", "source_type"]].drop_duplicates()
source_entities.columns = ["id", "label"]
target_entities = knowledge_graph[["target_id", "target_type"]].drop_duplicates()
target_entities.columns = ["id", "label"]
source_target_entities = pd.concat([source_entities, target_entities]).drop_duplicates()

valid_entities = pd.merge(source_target_entities, entities, on=["id", "label"], how="left", indicator=True)
knowledge_graph_entities_file = os.path.join(temp_dir, "knowledge_graph_entities.tsv")
valid_entities.to_csv(knowledge_graph_entities_file, sep="\t", index=False)

#### Check whether there are relationship types in the formatted_relation_type column that do not start with "BioMedGPS:"

### Copy all files to the dataset folder

In [22]:
dataset_name = "biomedgps"
dataset_version = "v20260406"
repo_commit_id = "f16e674d541170d5a296a35c1d16fab8c5f4c72a"
short_commit_id = repo_commit_id[:7]
outputdir = os.path.join(
    "/Volumes/T7",
    f"{dataset_name}-{dataset_version}-{short_commit_id}"
)
os.makedirs(outputdir, exist_ok=True)

print("Output dir:", outputdir)

Output dir: /Volumes/T7/biomedgps-v20260406-f16e674


In [23]:
os.makedirs(outputdir, exist_ok=True)

files = [
    (entity_file, os.path.join(outputdir, "annotated_entities.tsv")),
    (kg_file, os.path.join(outputdir, "knowledge_graph.tsv")),
    (annotated_kg_file, os.path.join(outputdir, "annotated_knowledge_graph.tsv")),
    (knowledge_graph_entities_file, os.path.join(outputdir, "knowledge_graph_entities.tsv")),
]

In [24]:
for f, output_file in files:
    print("Copying {} to {}".format(f, output_file))
    subprocess.check_output(["cp", f, output_file])
    output_zip_file = output_file + ".zip"
    print("Zipping {} to {}".format(output_file, output_zip_file))
    subprocess.check_output(["zip", "-j", output_zip_file, output_file])
    print("Removing {}".format(output_file))
    subprocess.check_output(["rm", output_file])

print("Please found all files in {}".format(outputdir))

Copying /Users/zhuzhixing/KG/biomedgps-data/graph_data/entities.tsv to /Volumes/T7/biomedgps-v20260406-f16e674/annotated_entities.tsv
Zipping /Volumes/T7/biomedgps-v20260406-f16e674/annotated_entities.tsv to /Volumes/T7/biomedgps-v20260406-f16e674/annotated_entities.tsv.zip
Removing /Volumes/T7/biomedgps-v20260406-f16e674/annotated_entities.tsv
Copying /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmp29yfyc32/knowledge_graph.tsv to /Volumes/T7/biomedgps-v20260406-f16e674/knowledge_graph.tsv
Zipping /Volumes/T7/biomedgps-v20260406-f16e674/knowledge_graph.tsv to /Volumes/T7/biomedgps-v20260406-f16e674/knowledge_graph.tsv.zip
Removing /Volumes/T7/biomedgps-v20260406-f16e674/knowledge_graph.tsv
Copying /var/folders/fp/m7jlbzzj4wldhdvbmlxw2h140000gn/T/tmp29yfyc32/annotated_knowledge_graph.tsv to /Volumes/T7/biomedgps-v20260406-f16e674/annotated_knowledge_graph.tsv
Zipping /Volumes/T7/biomedgps-v20260406-f16e674/annotated_knowledge_graph.tsv to /Volumes/T7/biomedgps-v20260406-f16e674/anno

### Upload the dataset to Dropbox and Dataverse

You can upload the dataset to Dropbox first, then select the files from Dropbox to upload to Dataverse.

https://dataverse.harvard.edu/dataverse/biomedgps